# Day 3 — KV Cache: Cached vs Uncached Decoding

**50-Day ML Infrastructure & LLM Systems Roadmap**

This notebook compares manual autoregressive decoding with and without `past_key_values` in DistilGPT-2.


## 1. Setup


In [ ]:
!pip install -q transformers accelerate psutil pandas matplotlib


In [ ]:
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
memory_info = psutil.virtual_memory()

environment_info = {
    "device": str(device).upper(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "total_runtime_ram_gb": memory_info.total / 1024**3,
    "available_ram_before_loading_gb": memory_info.available / 1024**3,
}

for key, value in environment_info.items():
    print(f"{key}: {value}")


## 2. Load DistilGPT-2


In [ ]:
MODEL_NAME = "distilbert/distilgpt2"
process = psutil.Process()
load_start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

load_time_seconds = time.perf_counter() - load_start
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

parameter_count = sum(p.numel() for p in model.parameters())
estimated_parameter_memory_mb = parameter_count * next(model.parameters()).element_size() / 1024**2

model_info = {
    "model_name": MODEL_NAME,
    "parameter_count": parameter_count,
    "estimated_parameter_memory_mb": estimated_parameter_memory_mb,
    "load_time_seconds": load_time_seconds,
    "process_ram_after_loading_mb": process.memory_info().rss / 1024**2,
}

for key, value in model_info.items():
    print(f"{key}: {value}")


## 3. Configuration and KV Cache Inspection


In [ ]:
with torch.inference_mode():
    prefill_output = model(
        input_ids=input_ids,
        use_cache=True,
    )

past_key_values = prefill_output.past_key_values


def extract_cache_tensors(cache):
    """Recursively collect tensors from legacy and modern cache formats."""
    tensors = []

    if torch.is_tensor(cache):
        tensors.append(cache)

    elif cache is None:
        pass

    elif isinstance(cache, (tuple, list)):
        for item in cache:
            tensors.extend(extract_cache_tensors(item))

    elif hasattr(cache, "layers"):
        for layer in cache.layers:
            if hasattr(layer, "keys"):
                tensors.extend(extract_cache_tensors(layer.keys))

            if hasattr(layer, "values"):
                tensors.extend(extract_cache_tensors(layer.values))

    return tensors


cache_tensors = extract_cache_tensors(past_key_values)

cache_tensor_count = len(cache_tensors)
cache_element_count = sum(
    tensor.numel()
    for tensor in cache_tensors
)

cache_memory_bytes = sum(
    tensor.numel() * tensor.element_size()
    for tensor in cache_tensors
)

cache_memory_mb = cache_memory_bytes / 1024**2

print(f"Cache type: {type(past_key_values).__name__}")
print(f"Cached tensors: {cache_tensor_count}")
print(f"Cached elements: {cache_element_count:,}")
print(f"Estimated initial KV cache memory: {cache_memory_mb:.4f} MB")

## 4. Manual Cached and Uncached Decoding


In [ ]:
def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize()


def generate_without_cache(prompt, max_new_tokens):
    generated_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)
    step_latencies = []
    generated_token_ids = []

    with torch.inference_mode():
        for _ in range(max_new_tokens):
            synchronize_device()
            start = time.perf_counter()
            outputs = model(input_ids=generated_ids, use_cache=False)
            next_token_id = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
            synchronize_device()
            step_latencies.append(time.perf_counter() - start)
            generated_token_ids.append(next_token_id.item())
            if next_token_id.item() == tokenizer.eos_token_id:
                break

    total_time = sum(step_latencies)
    return {
        "text": tokenizer.decode(generated_ids[0], skip_special_tokens=True),
        "generated_token_ids": generated_token_ids,
        "generated_tokens": len(generated_token_ids),
        "step_latencies": step_latencies,
        "total_time": total_time,
        "tokens_per_second": len(generated_token_ids) / total_time if total_time else 0.0,
    }


def generate_with_cache(prompt, max_new_tokens):
    prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)
    synchronize_device()
    prefill_start = time.perf_counter()

    with torch.inference_mode():
        outputs = model(input_ids=prompt_ids, use_cache=True)

    synchronize_device()
    prefill_latency = time.perf_counter() - prefill_start
    cache = outputs.past_key_values
    next_token_id = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
    generated_token_ids = [next_token_id.item()]
    decode_step_latencies = []
    current_token = next_token_id

    with torch.inference_mode():
        for _ in range(max_new_tokens - 1):
            if current_token.item() == tokenizer.eos_token_id:
                break
            synchronize_device()
            start = time.perf_counter()
            outputs = model(
                input_ids=current_token,
                past_key_values=cache,
                use_cache=True,
            )
            next_token_id = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
            cache = outputs.past_key_values
            current_token = next_token_id
            synchronize_device()
            decode_step_latencies.append(time.perf_counter() - start)
            generated_token_ids.append(next_token_id.item())

    generated_ids = torch.cat([
        prompt_ids,
        torch.tensor([generated_token_ids], device=device, dtype=prompt_ids.dtype),
    ], dim=-1)
    decode_time = sum(decode_step_latencies)
    total_time = prefill_latency + decode_time

    return {
        "text": tokenizer.decode(generated_ids[0], skip_special_tokens=True),
        "generated_token_ids": generated_token_ids,
        "generated_tokens": len(generated_token_ids),
        "prefill_latency": prefill_latency,
        "decode_step_latencies": decode_step_latencies,
        "decode_time": decode_time,
        "total_time": total_time,
        "tokens_per_second": len(generated_token_ids) / total_time if total_time else 0.0,
    }


## 5. Verify Identical Greedy Output


In [ ]:
uncached_example = generate_without_cache(PROMPT, MAX_NEW_TOKENS)
cached_example = generate_with_cache(PROMPT, MAX_NEW_TOKENS)
outputs_match = uncached_example["generated_token_ids"] == cached_example["generated_token_ids"]

print(f"Outputs match: {outputs_match}")
print()
print("Generated text:")
print(cached_example["text"])


## 6. Benchmark KV Cache Benefit


In [ ]:
def benchmark_method(method):
    function = generate_with_cache if method == "cached" else generate_without_cache
    for _ in range(WARMUP_RUNS):
        function(PROMPT, MAX_NEW_TOKENS)

    latencies = []
    throughputs = []
    step_latencies = []
    prefill_latencies = []

    for _ in range(MEASURED_RUNS):
        result = function(PROMPT, MAX_NEW_TOKENS)
        latencies.append(result["total_time"])
        throughputs.append(result["tokens_per_second"])
        if method == "cached":
            prefill_latencies.append(result["prefill_latency"])
            step_latencies.extend(result["decode_step_latencies"])
        else:
            step_latencies.extend(result["step_latencies"])

    return {
        "method": method,
        "mean_latency_seconds": np.mean(latencies),
        "median_latency_seconds": np.median(latencies),
        "minimum_latency_seconds": np.min(latencies),
        "maximum_latency_seconds": np.max(latencies),
        "latency_std_seconds": np.std(latencies),
        "mean_tokens_per_second": np.mean(throughputs),
        "mean_step_latency_seconds": np.mean(step_latencies),
        "mean_prefill_latency_seconds": np.mean(prefill_latencies) if prefill_latencies else np.nan,
    }

benchmark_df = pd.DataFrame([
    benchmark_method("uncached"),
    benchmark_method("cached"),
])
benchmark_df.round(4)


In [ ]:
uncached_row = benchmark_df[benchmark_df["method"] == "uncached"].iloc[0]
cached_row = benchmark_df[benchmark_df["method"] == "cached"].iloc[0]

latency_speedup = uncached_row["mean_latency_seconds"] / cached_row["mean_latency_seconds"]
latency_reduction_percent = (
    (uncached_row["mean_latency_seconds"] - cached_row["mean_latency_seconds"])
    / uncached_row["mean_latency_seconds"] * 100
)
throughput_improvement_percent = (
    (cached_row["mean_tokens_per_second"] - uncached_row["mean_tokens_per_second"])
    / uncached_row["mean_tokens_per_second"] * 100
)

print(f"Latency speedup: {latency_speedup:.2f}x")
print(f"Latency reduction: {latency_reduction_percent:.2f}%")
print(f"Throughput improvement: {throughput_improvement_percent:.2f}%")


## 7. Visualize the Difference


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    range(1, len(uncached_example["step_latencies"]) + 1),
    uncached_example["step_latencies"],
    marker="o",
    label="Uncached",
)
plt.plot(
    range(2, len(cached_example["decode_step_latencies"]) + 2),
    cached_example["decode_step_latencies"],
    marker="o",
    label="Cached",
)
plt.xlabel("Generated token step")
plt.ylabel("Latency in seconds")
plt.title("Per-Token Latency: Cached vs Uncached")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(benchmark_df["method"], benchmark_df["mean_latency_seconds"])
plt.ylabel("Mean total latency in seconds")
plt.title("Total Generation Latency")
plt.show()
